[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/structure-property_phi-sweep.ipynb)

# Structure-Property: Transverse Modulus vs. Fibre Volume Fraction

A structure-property study with a twist: instead of sweeping the fibre volume fraction $\phi$ of a
random-fibre RVE (`generation.rve.make_random_composite_rve`,
[Catalanotti 2016](https://doi.org/10.1016/j.compstruct.2015.11.039)) on an evenly-spaced grid,
this notebook picks which $\phi$ to actually run an (expensive) FFT solve at **actively** --
seed a few points, fit a Gaussian-process surrogate (`learning.surrogates.GPSurrogate`,
[GPJax](https://docs.jaxgaussianprocesses.com/) underneath) to what's been simulated so far, then
run the *next* simulation wherever that surrogate is currently most uncertain. Exactly
`scripts/active_learning.py`'s uncertainty-sampling loop, applied to $\phi$ instead of a VAE
latent code.

Each simulated point solves for the effective transverse Young's modulus $E(\phi)$ via
`learning.extractors.effective_modulus` (a mixed strain/stress boundary-condition displacement
solve -- same free-lateral-surface tensile test as
[Linear-Elastic Solve (mixed BC)](https://choROPeNt.github.io/FFTjax/documentation/examples/lin-elastic-mixed-bc)).
Unlike a calibration against already-digitized data, every new point here costs a real FFT
solve -- so uncertainty sampling isn't just a nicer curve, it's fewer expensive solves for the
same model quality.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Active learning: pick $\phi$, run the solve, refit, repeat

Same random-fibre generator as
[pff-damage.ipynb](https://choROPeNt.github.io/FFTjax/documentation/examples/phase-field),
here with no interphase (binary matrix/fibre) since only the effective elastic modulus is wanted.
One fixed `seed` across all simulations isolates $\phi$'s effect on $E$ from packing-realisation
noise.

Boundary condition per $\phi$: $\varepsilon_{11}$ strain-controlled (small uniaxial probe strain),
$\varepsilon_{22}, \varepsilon_{33}$ stress-controlled to zero (free lateral surfaces) -- a real
transverse tensile test, exactly `lin-elastic_mixed-BC.ipynb`'s `control`/`stress_goal`
convention. The resulting $E = \bar\sigma_{11} / \bar\varepsilon_{11}$ is the true engineering
transverse modulus, not the stiffer constrained coefficient a pure-strain BC would give.

The loop itself:

1. **Seed** with 3 real solves bracketing the range (both ends plus the midpoint) -- `GPSurrogate`
   needs at least a couple of points before "uncertain" means anything.
2. Fit the surrogate on every $(\phi, E)$ pair simulated so far.
3. Evaluate its predictive variance over a fine candidate grid spanning the range, **excluding**
   candidates within `MIN_GAP` of an already-simulated $\phi$ -- a noisy GP's posterior variance
   doesn't collapse to exactly zero at an observed point (and can even come back up near the
   domain's own edges), so without this guard the acquisition step can re-pick a $\phi$ that's
   already known and waste a real solve confirming it -- exactly what the first, unguarded version
   of this loop did.
4. Run a **real** FFT solve at the highest-variance remaining candidate.
5. Repeat for `N_ROUNDS` additions.

`N_ROUNDS` is exposed right below so you can see how few actively-placed solves it actually takes
before the surrogate stops changing much.

In [ ]:
from generation.rve import make_random_composite_rve
from materialmodels.elastic.isotropic import LinearElasticIsotropic
from learning.extractors import effective_modulus
from learning.surrogates import GPSurrogate

r_fiber   = 0.005   # mm
vox       = 0.001   # mm -- 5 voxels per fibre radius
size_in_r = 10       # domain side ~ 10*r_fiber (Catalanotti 2016 convention)
seed      = 42       # fixed packing realisation across the sweep

E_matrix, nu_matrix = 3500.0, 0.35    # epoxy matrix
E_fiber,  nu_fiber  = 70000.0, 0.20   # glass fibre
eps0 = 1.0e-3   # small uniaxial tensile strain probe (xx)

PHI_MIN, PHI_MAX = 0.10, 0.60
N_ROUNDS = 9      # MODIFY ME: actively-chosen solves to run, beyond the 3 seed points

phi_grid = np.linspace(PHI_MIN, PHI_MAX, 200)   # continuous search space, discretised for the acquisition step
MIN_GAP  = 0.02   # exclude candidates this close to an already-simulated phi (see note below)


def simulate(phi):
    phase_np, n, L, phi_act, _ = make_random_composite_rve(
        phi=phi, r_fiber=r_fiber, dx=vox, size_in_r=size_in_r, nz=1, K=15, seed=seed,
    )
    phase = jnp.array(phase_np.reshape(-1))
    materials = [
        LinearElasticIsotropic(E=E_matrix, nu=nu_matrix, name="epoxy matrix"),
        LinearElasticIsotropic(E=E_fiber, nu=nu_fiber, name="glass fibre"),
    ]
    E_val, nu_val, converged = effective_modulus(
        phase, materials, n, L, component=(0, 0), eps0=eps0, toler_lin=1e-6, maxiter=200,
    )
    return phi_act, E_val, nu_val[1], converged, n


phi_obs, E_obs, nu_obs = [], [], []
al_snapshots = []   # per round: n_obs, mean/std over phi_grid -- see the plot right after this cell

# Seed: bracket the range plus its midpoint, before any uncertainty-driven
# choice is even possible.
for phi in (PHI_MIN, 0.5 * (PHI_MIN + PHI_MAX), PHI_MAX):
    phi_act, E_val, nu_val, converged, n = simulate(phi)
    phi_obs.append(phi_act); E_obs.append(E_val); nu_obs.append(nu_val)
    print(f"[seed]        phi={phi_act:.3f}  grid={n}  E={E_val:8.1f} MPa  nu={nu_val:.4f}  converged={converged}")

for round_i in range(1, N_ROUNDS + 1):
    surrogate = GPSurrogate(num_iters=300).fit(phi_obs, E_obs, verbose=False)
    mean_grid, var_grid = surrogate.predict(phi_grid)
    al_snapshots.append({"n_obs": len(phi_obs), "mean_grid": mean_grid, "std_grid": np.sqrt(var_grid)})

    # Exclude candidates too close to an already-simulated phi -- a noisy GP's
    # posterior variance doesn't collapse all the way to zero at an observed
    # point, and can even come back UP near the domain's own edges (less
    # flanking data on one side) even after that edge has already been
    # sampled. Without this, argmax can re-select an already-known phi and
    # waste a real FFT solve confirming what's already known -- verified by
    # hitting exactly this on the first (unguarded) run of this cell.
    too_close = np.min(np.abs(phi_grid[:, None] - np.array(phi_obs)[None, :]), axis=1) < MIN_GAP
    var_search = np.where(too_close, -np.inf, var_grid)

    next_local = int(np.argmax(var_search))
    next_phi = float(phi_grid[next_local])
    phi_act, E_val, nu_val, converged, n = simulate(next_phi)
    phi_obs.append(phi_act); E_obs.append(E_val); nu_obs.append(nu_val)
    print(f"[{round_i:2d}/{N_ROUNDS}]  picked phi={next_phi:.3f} (GP std={np.sqrt(var_search[next_local]):.1f} MPa)  "
          f"-> phi_act={phi_act:.3f}  E={E_val:8.1f} MPa  nu={nu_val:.4f}  converged={converged}")

# one final snapshot, after the last addition
surrogate = GPSurrogate(num_iters=300).fit(phi_obs, E_obs, verbose=False)
mean_grid, var_grid = surrogate.predict(phi_grid)
al_snapshots.append({"n_obs": len(phi_obs), "mean_grid": mean_grid, "std_grid": np.sqrt(var_grid)})

phi_obs, E_obs, nu_obs = np.array(phi_obs), np.array(E_obs), np.array(nu_obs)

## Visualize: the surrogate tightening, and where it chose to look

Left: every GP snapshot's mean curve (darker = more observations), the final snapshot's $\pm2\sigma$
band, and every actual FFT solve -- seed points and actively-chosen ones marked separately. Right:
every actively-chosen $\phi$ plotted against the round it was picked in -- a histogram would be too
coarse at only a handful of points; this instead shows directly whether the search kept spreading
out or started revisiting the same neighbourhood.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

cmap = plt.cm.Blues
n_snap = len(al_snapshots)
for i, snap in enumerate(al_snapshots):
    color = cmap(0.35 + 0.55 * (i + 1) / n_snap)
    axes[0].plot(phi_grid, snap["mean_grid"], "-", color=color, label=f"GP after {snap['n_obs']} obs")

final = al_snapshots[-1]
axes[0].fill_between(phi_grid, final["mean_grid"] - 2 * final["std_grid"],
                      final["mean_grid"] + 2 * final["std_grid"],
                      color=cmap(0.9), alpha=0.15, label=r"final GP $\pm2\sigma$")
axes[0].plot(phi_obs[:3], E_obs[:3], "s", color="black", markersize=7, label="seed solves")
axes[0].plot(phi_obs[3:], E_obs[3:], "o", color="C3", markersize=6, label="actively chosen solves")
axes[0].set_xlabel(r"$\phi$ (fibre volume fraction)")
axes[0].set_ylabel(r"$E_{11}$ [MPa]")
axes[0].set_title("GP surrogate tightening with actively-chosen solves")
axes[0].legend(fontsize=8)
axes[0].grid(True, linewidth=0.5, alpha=0.6)

axes[1].plot(phi_obs[:3], [0] * 3, "s", color="black", markersize=8, label="seed solves (round 0)")
axes[1].plot(phi_obs[3:], range(1, len(phi_obs) - 2), "o-", color="C3", markersize=7,
             label="actively chosen, in order")
axes[1].set_xlim(PHI_MIN - 0.02, PHI_MAX + 0.02)
axes[1].set_xlabel(r"$\phi$ (fibre volume fraction)")
axes[1].set_ylabel("round picked (0 = seed)")
axes[1].set_title("When/where the active learner chose to sample")
axes[1].legend(fontsize=8)
axes[1].grid(True, linewidth=0.5, alpha=0.6)

fig.tight_layout()
plt.show()

## Next steps

- Lower `N_ROUNDS` to see how few actively-chosen solves it actually takes before the surrogate
  stops changing, or raise it to watch it keep refining.
- Add a second search dimension (e.g. `r_fiber`, or the matrix/fibre modulus ratio) --
  `GPSurrogate` accepts a multi-dimensional `X` unchanged, same as `scripts/active_learning.py`'s
  4-D latent-space GP; the acquisition step just needs a grid over both dimensions instead of one.
- This notebook's loop and `scripts/active_learning.py`'s (same uncertainty-sampling idea, over a
  pool of pre-generated RVE patches instead of a continuous $\phi$) are now two hand-written
  copies of the same shape -- a surrogate, a "pick the most uncertain candidate" acquisition step,
  and a real experiment to run there. A shared active-learning wrapper generalizing that shape is
  the natural next extraction, the same way `GPSurrogate`/`effective_modulus` themselves were.
- Swap the isotropic glass fibre for `TransverseIsotropic` carbon fibre (as in `pff-damage.ipynb`)
  and compare the longitudinal vs. transverse modulus sweep.